# Cyclical EMA Strategy Research

Goal: 在周期性波动美股标的上，验证 EMA 趋势跟踪 + LightGBM gating 是否产生 Sharpe ≥ 1.0 且优于纯 EMA baseline ≥ +0.3 的策略。

Reference:
- Design: `docs/plans/2026-05-07-cyclical-ema-research-design.md`
- Impl plan: `docs/plans/2026-05-08-cyclical-ema-research-impl.md`

In [ ]:
# === Cell 0: Imports + Constants ===
from __future__ import annotations
import os, json, math, time, warnings
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import yfinance as yf
import requests
from bs4 import BeautifulSoup

import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
import optuna
import shap

from oxq.indicators.builtin import EMA, ATR, MFI, OBV
from oxq.indicators.hurst_exponent import HurstExponent
from oxq.indicators.annualized_volatility import AnnualizedVolatility
from oxq.indicators.rolling_volatility import RollingVolatility
from oxq.indicators.garch_volatility import GarchVolatility
from oxq.indicators.rolling_mdd import RollingMDD
from oxq.indicators.nday_return import NdayReturn

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42)

# --- Constants ---
PROJECT_ROOT = Path("examples/research/cyclical_ema").resolve()
CACHE_DIR    = PROJECT_ROOT / "cache"
OUTPUT_DIR   = PROJECT_ROOT / "outputs"
CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# 数据刷新 flag —— 默认 False，迭代时使用缓存；改 True 强制重拉
REFRESH_DATA = False

# 时间锚点（来自 design doc 第 2.3 节）
ANCHOR_DATE   = pd.Timestamp("2026-05-07")            # universe filter as-of
TRAIN_CUTOFF  = pd.Timestamp("2021-05-07")            # 训练/测试切分
HISTORY_START = pd.Timestamp("2010-01-01")            # 数据起点（覆盖 ≥10y 的 universe）

# Universe 过滤阈值
MIN_LISTING_YEARS_STRICT = 10                          # A 组
MIN_LISTING_YEARS_RELAX  = 7                           # B 组
MIN_DOLLAR_VOL_STRICT    = 5_000_000                   # A 组
MIN_DOLLAR_VOL_RELAX     = 2_000_000                   # B 组
HURST_WINDOW             = 100
HURST_THRESHOLD_STRICT   = 0.40
HURST_THRESHOLD_RELAX    = 0.50
ANN_VOL_WINDOW           = 60
ANN_VOL_THRESHOLD_STRICT = 0.25
ANN_VOL_THRESHOLD_RELAX  = 0.20

# Sample 标签
LABEL_RETURN_THRESHOLD   = 0.05                        # gross_return ≥ 5% → 1
MIN_HISTORY_BARS         = 252                         # t_buy 之前最少交易日数

# 模型 + 回测
RANDOM_STATE             = 42
N_OPTUNA_TRIALS          = 50
N_CV_FOLDS               = 5
SCORE_THRESHOLD_X        = 0.55
TOP_N                    = 10
ROUND_TRIP_BPS           = 20                          # 0.20%

# 行业映射
SECTOR_TO_SPDR = {
    "Technology":              "XLK",
    "Financial Services":      "XLF",
    "Energy":                  "XLE",
    "Healthcare":              "XLV",
    "Industrials":             "XLI",
    "Consumer Defensive":      "XLP",
    "Consumer Cyclical":       "XLY",
    "Utilities":               "XLU",
    "Basic Materials":         "XLB",
    "Real Estate":             "XLRE",
    "Communication Services":  "XLC",
}
MARKET_TICKERS = ["^GSPC", "^VIX"] + list(set(SECTOR_TO_SPDR.values()))

print(f"Project root: {PROJECT_ROOT}")
print(f"REFRESH_DATA: {REFRESH_DATA}")
print(f"Anchor date: {ANCHOR_DATE.date()}, Train cutoff: {TRAIN_CUTOFF.date()}")
